# 개별종목 조합A — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합A 피처 12개를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합A의 피처 값만 지정합니다.
FEATURE_COLUMNS = ('sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_bandwidth', 'bb_position', 'atr_ratio', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return', 'five_day_return')

from models.stock_experiment import evaluate_stock_models  # noqa: E402
from models.stock_ranking import add_probability_ranks  # noqa: E402
from scripts.run_stock_model_experiment import load_stock_model_dataset  # noqa: E402

dataset = load_stock_model_dataset(FEATURE_COLUMNS)
print("학습 기간:", dataset.frame["bas_dd"].min(), "~", dataset.frame["bas_dd"].max())
print("학습 행·종목:", len(dataset.frame), dataset.frame["code"].nunique())
print("조합A 피처:", list(dataset.feature_columns))

result = evaluate_stock_models(
    dataset,
    model_builders={MODEL_NAME: MODEL_BUILDER},
)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(result.outer_results.loc[:, fold_columns].round(4))

metric_columns = ["accuracy", "macro_f1", "down_recall", "core_harmonic_mean"]
display(result.outer_results.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

ranked = add_probability_ranks(result.oos_predictions)
cache_path = project_root / "data" / "raw" / f"stock_model_oos_{MODEL_NAME}.parquet"
ranked.to_parquet(cache_path, index=False)
print("OOS 확률·랭킹 저장:", cache_path)


학습 기간: 20100330 ~ 20240822
학습 행·종목: 172983 162
조합A 피처: ['sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_bandwidth', 'bb_position', 'atr_ratio', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return', 'five_day_return']


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130409,20130704,0.3899,0.3667,0.1853,0.2807
1,2,balanced,999,20140411,20140710,0.4726,0.3288,0.1102,0.2108
2,3,balanced,1248,20150420,20150715,0.3729,0.3728,0.3355,0.3595
3,4,balanced,1497,20160422,20160719,0.4137,0.3820,0.2156,0.3102
4,5,balanced,1746,20170424,20170721,0.4248,0.3501,0.2130,0.3029
5,6,balanced,1995,20180503,20180731,0.3827,0.3808,0.3153,0.3567
6,7,balanced,2243,20190513,20190805,0.4068,0.3337,0.1238,0.2217
7,8,balanced,2492,20200515,20200806,0.3624,0.3485,0.5518,0.4032
8,9,balanced,2741,20210517,20210809,0.4389,0.4066,0.3480,0.3942
9,10,balanced,2990,20220519,20220812,0.3423,0.3377,0.2353,0.2961


,OOS 폴드 평균
accuracy,0.3990
macro_f1,0.3647
down_recall,0.2791
core_harmonic_mean,0.3240


OOS 확률·랭킹 저장: C:\Users\Administrator\Alpha_Stack\data\raw\stock_model_oos_XGBoost.parquet
